In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics_regression, preprocess_inference_results

results_dir = PROJECT_ROOT / "results"

## SOFA score prediction results

In [2]:
def process_sofa_prediction_results(input_dir, **kwargs) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual"),
        expected_expr=pl.col("expected"),
        filter_ambiguous=pl.col("stop_reason") == Reason.GOT_TOKEN,
        sofa = True,
        **kwargs,
    )

In [ ]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.mimic_icu import SofaPredictionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = SofaPredictionDataset(path)
len(dataset)

In [3]:
gt = pl.read_csv("/home/hb101@hsg.privat/ethos-benchmark/src/ethos/tokenize/maps/mimic-iv_derived.csv.gz").select("stay_id", "first_day_sofa")

expectations = [1.0, 3.5, 5.5, 8.0, 10.5, 12.5, 15.0, 17.5, 19.5, 22.0]

for input_dir in (results_dir / Task.SOFA_PREDICTION).iterdir():
    df = process_sofa_prediction_results(input_dir)

    y_pred = sum(df[f"Q{i + 1}"] * e for i, e in enumerate(expectations))

    y_true = (
        df.join(
            gt,
            left_on="icu_stay_id",
            right_on="stay_id",
            how="left",
        )["first_day_sofa"]
    )

    res_full = compute_and_print_metrics_regression(y_true.to_numpy(), y_pred.to_numpy())
    print(f"MAE: {res_full['mae']:.3f} for {len(df)*df['counts'][0]} samples")
    print(f"95% CI: {res_full['ci'][0]:.3f} - {res_full['ci'][1]:.3f}")

MAE: 1.660 for 219940 samples
95% CI: 1.633 - 1.688
